# Detect Columns by using ML

---

In [1]:
import numpy as np
import pandas as pd
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functions import *
import scipy
import signal
import io
import cv2
import os
import glob
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from scipy.ndimage import uniform_filter1d

**Machine learning (ML)** is a branch of artificial intelligence. It allows a computer to learn from data and to improve decision making with experience.

---

## Using Random Forest

**L'Arbre de Décision (Decision Tree) :**
Imagine un jeu de "Qui est-ce ?". L'algorithme pose une série de questions par oui/non sur les caractéristiques (features) de tes données pour arriver à une conclusion. Par exemple : "La variance de cette colonne est-elle supérieure à 450 ?" -> Si oui, on va à droite ; si non, on va à gauche.

Le Random Forest repose sur l'apprentissage d'ensemble (Ensemble Learning), et plus précisément sur une technique appelée Bagging (Bootstrap Aggregating). L'algorithme va créer une "forêt" composée de dizaines, voire de centaines d'arbres de décision.

Pour classer une nouvelle colonne (Normale vs Défectueuse), la forêt fait passer les données de la colonne dans tous ses arbres. Chaque arbre vote. La classe qui obtient la majorité des votes l'emporte.

In [ ]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path)
            images.append(img)
            
    return images

: 

### Features that we use : 

- Moyenne `mean` : Valeur moyenne des intensités de la colonne. Luminosité globale de la colonne. Si trop ou trop basse peut indiquer anomalie.
- Ecart-type `std` : Dispersion autour de la moyenne. Colonne peut-être *noisy* si l'écart-type est très élevé.
- Min et Max : Si la colonne à des valeurs super hautes ou super basses c'est que l'amplificateur est défectueux (dixit Phlypo)
- Range `range` : Différence entre max-min
- Médian `median` : Valeur centrale. Comparaison avec la moyenne intéressant.
- Quantile `q25` `q75` : Quartile.
- Skewness `skewness` : Asymétrie 
- Kurtosis `kurtosis`: Applatissement 
- Energie `energy` : Energie d'un signal (dixit Signals and Systems)
- Entropie `entropy` : J'ai pas vrmt compris / Mesure du désordre ou de l'incertitude dans la distribution des intensités.
- Nombres de pics `num_peaks` : les pics dans la colonne si gros peut-être *fragmented*
- Moyenne du gradiant `mean_gradient` : Moyenne des différences entre pixels consécutifs

#### A faire : 
- Différence Spatiale (un peu comme mes autres méthodes)
- Différence Temporelle (pour les blinking)

In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter1d

def extract_image_features_vectorized(img_prev, img_curr, img_next):
    # 1. Conversion en float64 pour toute l'image d'un coup
    img_prev = img_prev.astype(np.float64)
    img_curr = img_curr.astype(np.float64)
    img_next = img_next.astype(np.float64)
    
    height, width = img_curr.shape[:2]
    
    # DÉTECTION DE LA RÉSOLUTION (Pour la fenêtre du LT_moy)
    if height < 700: 
        taille_fenetre = 18
    elif height < 1050: 
        taille_fenetre = 31
    else: 
        taille_fenetre = 24

    # --- Statistiques Globales ---
    col_mean = np.mean(img_curr, axis=0) # Vecteur avec la moyenne de chaque colonne
    col_energy = np.sum(img_curr ** 2, axis=0) / height
    
    # =========================================================================
    # 1. LT_moy (Fenêtre glissante ultra-rapide)
    # =========================================================================
    # uniform_filter1d fait la tendance locale pour TOUTES les colonnes instantanément
    tendance_locale = uniform_filter1d(col_mean, size=taille_fenetre, mode='reflect')
    
    # Calcul de l'écart-type local via la variance : V(X) = E(X^2) - E(X)^2
    mean_sq = np.mean(img_curr ** 2, axis=0)
    tendance_sq = uniform_filter1d(mean_sq, size=taille_fenetre, mode='reflect')
    std_locale = np.sqrt(np.maximum(tendance_sq - tendance_locale**2, 0))
    
    ecart_lt_moy = np.abs(col_mean - tendance_locale)
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5)
    
    # =========================================================================
    # 2. Spatial & Temporel
    # =========================================================================
    # Décalage du vecteur pour comparer avec la colonne de gauche et de droite
    left_neighbor = np.roll(col_mean, 1)
    right_neighbor = np.roll(col_mean, -1)
    neighbor_mean = (left_neighbor + right_neighbor) / 2.0
    
    # Correction des extrêmes (bords de l'image)
    neighbor_mean[0] = col_mean[1]
    neighbor_mean[-1] = col_mean[-2]
    
    spatial_diff = np.abs(col_mean - neighbor_mean)
    
    # Contexte temporel
    mean_prev = np.mean(img_prev, axis=0)
    mean_next = np.mean(img_next, axis=0)
    diff_temp_absolue = np.abs(col_mean - mean_prev)
    scintillement_temporel = np.abs(col_mean - ((mean_prev + mean_next) / 2.0))

    # --- Assemblage Final (Seulement nos 7 super-features !) ---
    features_matrix = np.column_stack((
        ecart_lt_moy, ratio_lt_moy,
        spatial_diff, diff_temp_absolue, scintillement_temporel,
        col_mean, col_energy
    ))
    
    return features_matrix

def process_single_image(img_num, images_list, json_data):
    """Prépare les labels et lance l'extraction de l'image entière"""
    img_curr = images_list[img_num]
    img_prev = images_list[img_num - 1] if img_num > 0 else img_curr
    img_next = images_list[img_num + 1] if img_num < (len(images_list) - 1) else img_curr
    
    width = img_curr.shape[1]
    
    # Extraction vectorisée
    features_matrix = extract_image_features_vectorized(img_prev, img_curr, img_next)
    
    # Création rapide des labels (y)
    defects = set(get_defect_coordinates(json_data, img_num))
    y_img = [1 if x in defects else 0 for x in range(width)]
    
    return features_matrix.tolist(), y_img

: 

In [ ]:
def build_dataset(images, json_data):
    print(f"Lancement de l'extraction sur {len(images)} images en parallèle...")
    
    # n_jobs=-1 (utilisation de tous les coeurs)
    # return_as="generator" permet à tqdm de se mettre à jour en temps réel
    result_generator = Parallel(n_jobs=-1, return_as="generator")(
        delayed(process_single_image)(img_num, images, json_data) 
        for img_num in range(len(images))
    )
    
    X = []
    y = []
    
    # On enveloppe le générateur avec tqdm pour la barre de progression
    for X_img, y_img in tqdm(result_generator, total=len(images), desc="Extraction Multicoeur"):
        X.extend(X_img)
        y.extend(y_img)
        
    return np.array(X), np.array(y)

: 

### Lezz gooo

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import traceback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm # Import modifié pour éviter l'erreur IProgress !

# =========================================================================
# 1. PARAMÈTRES DU SUPER DATASET
# =========================================================================
CAMERA_TYPE = 'HD'  # Change ceci en 'HD' ou 'SXGA' selon ce que tu veux entraîner

# Mapping pour faire correspondre le numéro de config au nom du dossier 'dyn'
dyn_mapping = {
    1: 'low dyn with columns 1', # Blinking
    2: 'low dyn with columns 2', # Noisy
    3: 'low dyn with columns 3'  # Noisy Blinking
}

X_list = []
y_list = []

# Noms de nos 7 features (Ordre strict correspondant à features_matrix)
noms_colonnes = [
    'ecart_lt_moy', 'ratio_lt_moy', 
    'spatial_diff', 'diff_temp_absolue', 'scintillement_temporel', 
    'mean', 'energy'
]

# =========================================================================
# 2. BOUCLE D'EXTRACTION AUTOMATIQUE (Les 9 combinaisons)
# =========================================================================
for seq in [1, 2, 3]:
    for config in [1, 2, 3]:
        print(f"\n🚀 Lancement : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        # Adapte le chemin vers ton dossier results si besoin
        # Le VRAI chemin absolu qui marche
        dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"
        json_path = os.path.join(dossier_json, f"{CAMERA_TYPE}_{sequence_name}_config_{config}.json") 
        
        try:
            # /!\ Assure-toi que load_images trouve bien les dossiers !
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            # Extraction multicœur avec la NOUVELLE fonction vectorisée
            X_batch, y_batch = build_dataset(images, json_data)
            
            X_list.append(X_batch)
            y_list.append(y_batch)
            
        except Exception as e:
            print(f"⚠️ Erreur ou fichier manquant pour Seq {seq} / Config {config} : {e}")
            traceback.print_exc()
            continue

# Fusion de tous les batchs en de grands tableaux Numpy
X_global = np.vstack(X_list)
y_global = np.concatenate(y_list)

print(f"\n✅ Extraction terminée ! Taille totale du dataset : {X_global.shape[0]} colonnes.")

# =========================================================================
# 3. RÉÉQUILIBRAGE DU SUPER DATASET
# =========================================================================
indices_defauts = np.where(y_global == 1)[0]
indices_sains = np.where(y_global == 0)[0]

print(f"Avant équilibrage : {len(indices_defauts)} défauts et {len(indices_sains)} colonnes saines.")

# On prend 2 fois plus de colonnes saines que de défauts pour un bon apprentissage
nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_balanced = X_global[indices_finaux]
y_balanced = y_global[indices_finaux]

print(f"Après équilibrage : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} colonnes saines.")

# =========================================================================
# 4. ENTRAÎNEMENT DU MODÈLE FINAL XGBOOST
# =========================================================================
X_balanced_df = pd.DataFrame(X_balanced, columns=noms_colonnes)
X_train, X_test, y_train, y_test = train_test_split(X_balanced_df, y_balanced, test_size=0.2, random_state=42)

print("\n🚀 Entraînement de XGBoost sur la RTX 2060 en cours...")

# Calcul du ratio pour l'équilibrage des classes dans XGBoost
ratio_poids = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

clf = xgb.XGBClassifier(
    n_estimators=150,          # Nombre d'arbres
    scale_pos_weight=ratio_poids, 
    random_state=42,
    tree_method="hist",        # Obligatoire pour le GPU
    device="cuda"              # Utilise la VRAM de la RTX 2060 !
)

print("Shape de X_train avant entrainement :", np.array(X_train).shape)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n📊 RÉSULTATS DU MODÈLE XGBOOST :")
print(classification_report(y_test, y_pred))

print("\n🏆 TOP 5 DES CARACTÉRISTIQUES LES PLUS IMPORTANTES :")
importances = clf.feature_importances_
for name, importance in sorted(zip(noms_colonnes, importances), key=lambda x: x[1], reverse=True)[:5]:
    print(f"{name}: {importance:.3f}")

: 

In [ ]:
# Optionnel : On crée un dossier 'models' pour garder le répertoire propre
os.makedirs('models', exist_ok=True)

# Définition du chemin de sauvegarde
model_path = 'models/xgboost_HD_baseline.json'

# Sauvegarde du modèle XGBoost
clf.save_model(model_path)

print(f"✅ Modèle HD sauvegardé avec succès sous : {model_path}")

: 

In [ ]:
import os

# Remets ton chemin absolu ici
dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"

print(f"1. Le dossier existe-t-il ? {os.path.exists(dossier_json)}")

if os.path.exists(dossier_json):
    fichiers = os.listdir(dossier_json)
    print(f"2. Python voit {len(fichiers)} fichiers dans ce dossier.")
    print(f"3. Voici les 5 premiers fichiers qu'il voit :")
    for f in fichiers[:5]:
        print(f"   -> '{f}'")

: 